**Setting**: Running on WSL:Ubuntu  
**Creating env**: `source "path"/venv/bin/activate`  
**Setup virtual env**:  
`python -m venv venv`  
`source venv/bin/activate`  
`pip install pyspark==4.0.1 kagglehub confluent-kafka jupyterlab`  
**Running docker**: `docker compose up -d`  
**Checking status cont**: `docker ps`



In [ ]:
from pyspark.sql import SparkSession

# Creating Spark Session
spark = (SparkSession.builder
    .appName("BigData_Lab01_Ubuntu") # Spark session name
    .master("local[*]") 
    .config("spark.driver.memory", "4g")
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
            "org.apache.kafka:kafka-clients:3.6.0,"
            "org.apache.spark:spark-streaming-kafka-0-10_2.13:4.0.1") 
    .getOrCreate())

# Checking success
print(f"Successfully! Spark version: {spark.version}")

Thành công! Spark version: 4.0.1


In [ ]:
import kagglehub

# Download dataset from kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# reading data to spark frame
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

# Checking by show 1 row of each df
df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row
+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row
+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row


In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

# Connect to brokers
admin_client = AdminClient({'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'})

# Setting
new_topics = [
    NewTopic(topic="ratings", num_partitions=1, replication_factor=3),
    NewTopic(topic="movies", num_partitions=1, replication_factor=3),
    NewTopic(topic="tags", num_partitions=1, replication_factor=3)
]

# Creating topics
fs = admin_client.create_topics(new_topics)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' created.")
    except Exception as e:
        print(f"Error when creating topic {topic}: {e}")

Lỗi khi tạo topic ratings: KafkaError{code=TOPIC_ALREADY_EXISTS,val=36,str="Topic 'ratings' already exists."}
Lỗi khi tạo topic movies: KafkaError{code=TOPIC_ALREADY_EXISTS,val=36,str="Topic 'movies' already exists."}
Lỗi khi tạo topic tags: KafkaError{code=TOPIC_ALREADY_EXISTS,val=36,str="Topic 'tags' already exists."}


In [ ]:
from pyspark.sql.functions import struct, to_json

# Kafka bootstrap servers
brokers = "localhost:9092,localhost:9192,localhost:9292"

def push_to_kafka(df, topic_name):
    df.selectExpr("to_json(struct(*)) AS value") \
      .write \
      .format("kafka") \
      .option("kafka.bootstrap.servers", brokers) \
      .option("topic", topic_name) \
      .save()

push_to_kafka(df_ratings, "ratings")
push_to_kafka(df_movies, "movies")
push_to_kafka(df_tags, "tags")

print("Data push to kafka successfully!")

Dữ liệu đã được nạp thành công vào Kafka!


In [9]:
from pyspark.sql.functions import from_json, col, avg, count
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

# Schema rating
rating_schema = StructType([
    StructField("userId", IntegerType()),
    StructField("movieId", IntegerType()),
    StructField("rating", DoubleType()),
    StructField("timestamp", IntegerType())
])

tag_schema = StructType([
    StructField("userId", IntegerType()),
    StructField("movieId", IntegerType()),
    StructField("tag", StringType()),
    StructField("timestamp", IntegerType())
])

*Problem 1*: Get data form kafka

In [ ]:
# Read from kafka
df_stream_ratings = (spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292")
    .option("subscribe", "ratings")
    .option("startingOffsets", "earliest") # Đọc từ đầu dữ liệu đã push
    .load())

df_stream_tags = (spark.read
                  .format("kafka")
                  .option("kafka.bootstrap.servers", "localhost:9092, localhost:9192, localhost:9292")
                  .option("subscribe", "tags")
                  .option("startingOffsets", "earliest")
                  .load())

# Parsed df
df_ratings_parsed = df_stream_ratings.select(
    from_json(col("value").cast("string"), rating_schema).alias("data")
).select("data.*")

df_tags_parsed = df_stream_tags.select(
    from_json(col("value").cast("string"), tag_schema).alias("data")
).select("data.*")

*Problem 2*: Get top 5 movies by avg rating, rating count higher than 30

In [ ]:
df_top_result = df_ratings_parsed.groupBy("movieId") \
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("rating_count")
    ) \
    .filter(col("rating_count") > 30) \
    .orderBy(col("avg_rating").desc()) \
    .limit(5)

# 2. Join với df_movies để lấy Title
# Lưu ý: df_movies bạn đã load ở cell trước đó
df_final_top_5 = df_top_result.join(df_movies, "movieId") \
    .select("title", "avg_rating", "rating_count") \
    .orderBy(col("avg_rating").desc())

df_final_top_5.show(truncate=False)

+---------------------------------------------+-----------------+------------+
|title                                        |avg_rating       |rating_count|
+---------------------------------------------+-----------------+------------+
|Streetcar Named Desire, A (1951)             |4.475            |40          |
|Shawshank Redemption, The (1994)             |4.429022082018927|634         |
|Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)|4.333333333333333|54          |
|Hustler, The (1961)                          |4.333333333333333|36          |
|Double Indemnity (1944)                      |4.323529411764706|34          |
+---------------------------------------------+-----------------+------------+



*Problem 3*: Get 5 worst tags

In [15]:
df_worst_tag = df_tags_parsed.join(df_ratings_parsed,["movieId", "userId"])\
                                .groupBy("tag") \
                                .agg(
                                    avg("rating").alias("avg_rating")
                                )\
                                .orderBy(col("avg_rating").asc())\
                                .limit(5)\
                                .select("tag", "avg_rating")

df_worst_tag.show(truncate=False)


+---------------+----------+
|tag            |avg_rating|
+---------------+----------+
|Ghosts         |0.5       |
|symbolic       |0.5       |
|HORRIBLE ACTING|0.5       |
|boring         |0.75      |
|camp           |1.0       |
+---------------+----------+



*Problem 4*: Check movie ratings

In [17]:
worst_tags_list = [row.tag for row in df_worst_tag.collect()]

df_analysis = df_ratings_parsed.join(df_tags_parsed, ["movieId", "userId"])\
                            .join(df_movies, "movieId")\
                            .filter(col("tag").isin(worst_tags_list))

analysis = df_analysis.groupBy("tag", "title")\
                    .agg(
                        avg("rating").alias("movie_avg_rating"),
                        count("rating").alias("num_ratings_of_this_movie")
                    )\
                    .orderBy("tag", "movie_avg_rating")

analysis.show(truncate=False)

+---------------+----------------------+----------------+-------------------------+
|tag            |title                 |movie_avg_rating|num_ratings_of_this_movie|
+---------------+----------------------+----------------+-------------------------+
|Ghosts         |Ghostbusters II (1989)|0.5             |4                        |
|HORRIBLE ACTING|Invisible, The (2007) |0.5             |4                        |
|boring         |Begotten (1990)       |0.5             |4                        |
|boring         |Twilight (2008)       |1.0             |4                        |
|camp           |Meatballs (1979)      |1.0             |4                        |
|symbolic       |Begotten (1990)       |0.5             |4                        |
+---------------+----------------------+----------------+-------------------------+

